# NB09 — Musanga cover versus deforestation age on native disturbance grids
V5 · CPU · Run 00–03 sequentially. Replaces the previous 50 m exploratory notebook (retained in Git history at bac1092ab241bb4c2f4e9e89083995ac0e520c57). No HLS extraction, model training or NB08 dependency. Uses NB07 final binary canopy maps, the original ROI eroded 100 m, Hansen v1.13 and JRC TMF v2025. Each dataset retains its own actual native CRS, pixel dimensions and lattice origin, checked from Earth Engine source bands (not mosaic default projection). UAV cover estimated by dense 30×30 subpixel sampling per native pixel (~1 m); ≥90% valid support inside ROI. Overlapping mosaics: select highest support, deterministic name tie-break; no double counting. Dates strictly before UAV survey calendar year. No detected event is not age zero or proof of undisturbed forest. JRC deforestation age = elapsed time since latest transition from mapped forest (1,2,4) to deforested land (3); persistent class 3 does not reset the clock. Regrowth age = elapsed time since onset of current continuous class 4 episode following mapped deforestation (3). Other transitions and left-censored histories are reported, not assigned invented ages. Retrospective classifications can use later imagery. Spatial chronosequence, not longitudinal succession or causal evidence.
Sources: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2025_v1_13 ; https://forobs.jrc.ec.europa.eu/TMF/resources/tutorial/gee .

In [ ]:
#@title 00 — Mount Drive, verify UAV inputs and study region
%pip -q install earthengine-api rasterio geopandas pyarrow scipy pandas matplotlib
from pathlib import Path
import os,json,hashlib,math,time
from collections import defaultdict
from contextlib import ExitStack
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio,ee
from rasterio import Affine
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from scipy.stats import spearmanr
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),Path('/content/drive/MyDrive/Musanga/Final/V5')] if (p.parent/'Data').is_dir() and p.is_dir()),None)
assert ROOT is not None
SITES=['Baego','MOS','Yangambi','Yoko'];YEARS=dict(Baego=2023,MOS=2024,Yangambi=2024,Yoko=2020)
PROJECT='musanga-511110';CRS='EPSG:32635';SUPPORT=.90;SUB=30;TILE=64
GID='UMD/hansen/global_forest_change_2025_v1_13';JID='projects/JRC/TMF/v1_2025/AnnualChanges'
FORCE_REBUILD=False
def digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str).encode()).hexdigest()
def atomic_json(p,x):
 tmp=p.with_suffix('.partial');tmp.write_text(json.dumps(x,indent=2,default=str));os.replace(tmp,p)
def atomic_parquet(p,x):
 tmp=p.with_suffix('.partial');x.to_parquet(tmp,index=False);os.replace(tmp,p)
NB07=ROOT/'Outputs/07_RGB_unified_production_v5';M=json.loads((NB07/'07_production_mosaic_manifest.json').read_text())
assert M['complete'] and M['schema']=='musanga_v5_nb07_rgb_mosaics' and len(M['mosaics'])==6
MOS=defaultdict(list);meta=[]
for item in M['mosaics']:
 p=NB07/'reconstructed_mosaics'/Path(item['output']).name
 assert p.is_file(),f'Missing NB07 mosaic: {p}'
 rec=json.loads((p.parent/(item['name']+'_reconstruction.json')).read_text())
 assert rec['status']=='complete' and rec['digest']==item['digest'] and rec['model']==M['model_sha'] and rec['policy']==M['policy_sha']
 with rasterio.open(p) as ds:
  assert ds.count==2 and ds.nodata==-9999 and ds.crs==rasterio.crs.CRS.from_string(CRS)
  assert ds.tags().get('signature')==item['digest'] and ds.width==item['width'] and ds.height==item['height']
  assert ds.descriptions==('Musanga probability','Musanga binary at 0.5')
  b=list(ds.bounds)
 MOS[item['site']].append((item['name'],p,b));meta.append(dict(name=item['name'],site=item['site'],signature=item['digest'],size=p.stat().st_size,bounds=b))
assert set(MOS)==set(SITES)
for s in SITES:MOS[s].sort(key=lambda x:x[0])
raw=json.loads((ROOT/'Outputs/00_input_audit/00_input_manifest.json').read_text())
rp=Path(raw['roi'])
if not rp.is_file():rp=ROOT.parent/'Data/Ortho_RGB'/rp.name
assert rp.is_file()
roi=unary_union(gpd.read_file(rp).to_crs(CRS).geometry).buffer(-100.)
assert roi.is_valid and not roi.is_empty
GEOM={s:roi.intersection(unary_union([box(*b) for _,_,b in MOS[s]])) for s in SITES}
assert all(not g.is_empty for g in GEOM.values())
try:ee.Initialize(project=PROJECT)
except Exception:ee.Authenticate();ee.Initialize(project=PROJECT)
print('Verified six NB07 mosaics; no new UAV inference.')


In [ ]:
#@title 01 — Obtain and verify the original Hansen and JRC lattices
G=ee.Image(GID);J=ee.ImageCollection(JID)
def projinfo(img,band):
 p=img.select(band).projection().getInfo()
 a=Affine(*p['transform']);assert abs(a.b)<1e-12 and abs(a.d)<1e-12 and a.a>0 and a.e<0,p
 return p['crs'],a
def samegrid(p,q):
 c,a=p;d,b=q
 if c!=d or not np.allclose([a.a,a.b,a.d,a.e],[b.a,b.b,b.d,b.e],rtol=1e-9,atol=1e-12):return False
 x,y=(~a)*(b.c,b.f)
 return abs(x-round(x))<1e-5 and abs(y-round(y))<1e-5
PG=projinfo(G,'lossyear')
for band in ['datamask','treecover2000']:assert samegrid(PG,projinfo(G,band))
GRIDS={};IMAGES={};REGIONS={};TILES={}
for s in SITES:
 y=YEARS[s];g=GEOM[s]
 lonlat=gpd.GeoSeries([g],crs=CRS).to_crs('EPSG:4326').iloc[0]
 region=ee.Geometry.Rectangle(list(lonlat.bounds),proj='EPSG:4326',geodesic=False)
 jc=J.filterBounds(region).sort('system:index');ids=jc.aggregate_array('system:index').getInfo()
 assert ids,f'No JRC source tiles at {s}'
 bands=[f'Dec{k}' for k in range(1990,y)]
 js=jc.toList(len(ids));pj=projinfo(ee.Image(js.get(0)),bands[0])
 # Verify every selected annual source band on every intersecting original tile.
 for i in range(len(ids)):
  src=ee.Image(js.get(i));info=src.select(bands).getInfo()['bands']
  for b in info:
   assert samegrid(pj,(b['crs'],Affine(*b['crs_transform']))),(s,b)
 # Use source projection rather than default one-degree mosaic projection.
 jimg=jc.select(bands).mosaic().setDefaultProjection(pj[0],list(pj[1])[:6]).unmask(255).toUint8()
 land=G.select('datamask');loss=G.select('lossyear').unmask(0)
 gimg=ee.Image.cat([land.unmask(255),loss.updateMask(land.eq(1)).unmask(255),G.select('treecover2000').unmask(255)]).toUint8()
 for tag,p,img in [('Hansen',PG,gimg),('JRC',pj,jimg)]:
  c,a=p;GRIDS[s,tag]=p;IMAGES[s,tag]=img
  geom=gpd.GeoSeries([g],crs=CRS).to_crs(c).iloc[0];REGIONS[s,tag]=geom
  x0,y0,x1,y1=geom.bounds
  cl,rt=(~a)*(x0,y1);cr,rb=(~a)*(x1,y0)
  tiles=[]
  for row in range(math.floor(rt/TILE)*TILE,math.ceil(rb/TILE)*TILE,TILE):
   for col in range(math.floor(cl/TILE)*TILE,math.ceil(cr/TILE)*TILE,TILE):
    l,t=a*(col,row);r,b=a*(col+TILE,row+TILE)
    if geom.intersects(box(l,b,r,t)):tiles.append((row,col))
  assert tiles;TILES[s,tag]=tiles
  print(s,tag,c,list(a)[:6],len(tiles),'tiles')
CFG=dict(version='native_ecology_v1',sources=[GID,JID],years=YEARS,roi_sha=hashlib.sha256(roi.wkb).hexdigest(),mosaics=meta,model=M['model_sha'],policy=M['policy_sha'],support=SUPPORT,sub=SUB,tile=TILE,grids={f'{s}_{t}':[c,list(a)[:6]] for (s,t),(c,a) in GRIDS.items()})
OUT=ROOT/'Outputs/09_ecology_v5'/digest(CFG)[:20];OUT.mkdir(parents=True,exist_ok=True)
CACHE=OUT/'tiles';CACHE.mkdir(exist_ok=True);atomic_json(OUT/'config.json',CFG)
print('Native-grid output:',OUT)


In [ ]:
#@title 02 — Aggregate UAV cover directly to native pixels and attach disturbance histories
def native_tile(s,tag,row,col,stack):
 c,a=GRIDS[s,tag];tr=a*Affine.translation(col,row)
 # Approximate area integration by equally spaced subpixel centres within each native footprint.
 hi=tr*Affine.scale(1/SUB,1/SUB);size=TILE*SUB
 inside=rasterize([(REGIONS[s,tag],1)],out_shape=(size,size),transform=hi,fill=0,dtype='uint8').astype(bool)
 def reduce(z):return z.reshape(TILE,SUB,TILE,SUB).sum((1,3))
 best=np.zeros((TILE,TILE),dtype=np.int32);positive=best.copy();source=np.full(best.shape,-1,dtype=np.int16)
 for i,(name,p,b) in enumerate(MOS[s]):
  ds=stack[name]
  with WarpedVRT(ds,crs=c,transform=hi,width=size,height=size,resampling=Resampling.nearest,src_nodata=-9999,nodata=-9999,warp_mem_limit=128) as vrt:
   z=vrt.read(2)
  valid=inside&((z==0)|(z==1));nv=reduce(valid);npix=reduce(valid&(z==1))
  better=nv>best;best[better]=nv[better];positive[better]=npix[better];source[better]=i
 keep=best>=math.ceil(SUPPORT*SUB*SUB)
 if not keep.any():return pd.DataFrame()
 # Explicit PixelGrid affine; Earth Engine returns structured NPY, not a resampled arbitrary 30m UTM tile.
 grid=dict(dimensions=dict(width=TILE,height=TILE),affineTransform=dict(scaleX=tr.a,shearX=tr.b,translateX=tr.c,shearY=tr.d,scaleY=tr.e,translateY=tr.f),crsCode=c)
 for attempt in range(5):
  try:
   arr=ee.data.computePixels(dict(expression=IMAGES[s,tag],fileFormat='NUMPY_NDARRAY',grid=grid));break
  except Exception:
   if attempt==4:raise
   time.sleep(2**attempt)
 assert arr.shape==(TILE,TILE)
 rr,cc=np.where(keep);xx,yy=tr*(cc+.5,rr+.5)
 d=pd.DataFrame(dict(site=s,product=tag,row=rr+row,col=cc+col,x_native=xx,y_native=yy,survey_year=YEARS[s],support=best[keep]/(SUB*SUB),cover_pp=100*positive[keep]/best[keep],mosaic=[MOS[s][i][0] for i in source[keep]]))
 y=YEARS[s]
 if tag=='Hansen':
  assert set(arr.dtype.names)=={'datamask','lossyear','treecover2000'},arr.dtype.names
  mask=arr['datamask'][keep];loss=arr['lossyear'][keep].astype(int);tc=arr['treecover2000'][keep]
  ok=(mask==1)&(loss>=0)&(loss<=25)
  prior=ok&(loss>0)&(2000+loss<y)
  d['history_valid']=ok;d['loss_year']=np.where(prior,2000+loss,np.nan)
  d['age_since_loss']=y-d.loss_year
  d['loss_status']=np.select([~ok,prior,ok&(loss>0)&(2000+loss>=y)],['invalid','prior loss','survey year or later loss'],default='no detected loss')
  d['treecover2000']=np.where(tc<=100,tc,np.nan)
 else:
  years=np.arange(1990,y);names=[f'Dec{k}' for k in years]
  assert set(arr.dtype.names)==set(names),arr.dtype.names
  hist=np.stack([arr[b][keep] for b in names],axis=0)
  ok=np.isin(hist,np.arange(1,7)).all(axis=0)
  # Only genuine transitions from forest to class 3, not each year spent deforested.
  entries=np.isin(hist[:-1],[1,2,4])&(hist[1:]==3)
  latest=np.max(np.where(entries,years[1:,None],0),axis=0)
  # Current uninterrupted class 4 episode, with directly preceding deforested state.
  run=np.cumprod((hist[::-1]==4).astype(np.int16),axis=0).sum(axis=0)
  onset=len(years)-run;prev=hist[np.maximum(onset-1,0),np.arange(len(d))]
  confirmed=ok&(run>0)&(onset>0)&(prev==3)
  d['history_valid']=ok;d['jrc_state']=np.where(ok,hist[-1],np.nan)
  d['deforestation_year']=np.where(ok&(latest>0),latest,np.nan)
  d['age_since_deforestation']=y-d.deforestation_year
  d['regrowth_onset_year']=np.where(confirmed,years[0]+onset,np.nan)
  d['age_since_regrowth']=y-d.regrowth_onset_year
  d['deforestation_status']=np.select([~ok,latest>0,(hist==3).any(axis=0)],['invalid','dated forest-to-deforested transition','deforested state without dated forest transition'],default='no detected deforestation')
  d['regrowth_status']=np.select([~ok,confirmed,run>0],['invalid','dated post-deforestation regrowth','other or left-censored regrowth'],default='not currently regrowth')
 return d
all_tables={}
for s in SITES:
 with ExitStack() as es:
  stack={name:es.enter_context(rasterio.open(p)) for name,p,b in MOS[s]}
  for tag in ['Hansen','JRC']:
   chunks=[]
   for i,(row,col) in enumerate(TILES[s,tag]):
    p=CACHE/f'{s}_{tag}_{row}_{col}.parquet'
    if p.is_file() and not FORCE_REBUILD:d=pd.read_parquet(p)
    else:
     d=native_tile(s,tag,row,col,stack);atomic_parquet(p,d)
    if len(d):chunks.append(d)
    print(s,tag,i+1,'/',len(TILES[s,tag]),'pixels',len(d),flush=True)
   assert chunks,(s,tag)
   all_tables[s,tag]=pd.concat(chunks,ignore_index=True)
for tag in ['Hansen','JRC']:
 d=pd.concat([all_tables[s,tag] for s in SITES],ignore_index=True)
 assert not d.duplicated(['site','row','col']).any()
 assert d.cover_pp.between(0,100).all() and d.support.ge(SUPPORT).all()
 atomic_parquet(OUT/f'{tag}_native_pixels.parquet',d)
 display(d.groupby('site').agg(n=('cover_pp','size'),valid_history=('history_valid','sum'),mean_support=('support','mean')))


In [ ]:
#@title 03 — Four simple figures: raw pixel-level cover against age and JRC state
H=pd.read_parquet(OUT/'Hansen_native_pixels.parquet');J9=pd.read_parquet(OUT/'JRC_native_pixels.parquet')
rows=[]
for d,col,label in [(H,'age_since_loss','Years since Hansen detected tree-cover loss'),(J9,'age_since_deforestation','Years since JRC forest-to-deforested transition'),(J9,'age_since_regrowth','Years since onset of JRC post-deforestation regrowth')]:
 fig,axs=plt.subplots(2,2,figsize=(11,8),sharex=True,sharey=True,layout='constrained')
 maxage=max(1,d[col].max()) if d[col].notna().any() else 1
 for s,ax in zip(SITES,axs.ravel()):
  g=d[d.site.eq(s)];q=g[g.history_valid&g[col].notna()]
  x=q[col];y=q.cover_pp
  rho=spearmanr(x,y).statistic if len(q)>2 and x.nunique()>1 and y.nunique()>1 else np.nan
  ax.scatter(x,y,s=5,alpha=.18,rasterized=True)
  ax.set(title=f'{s} | n={len(q):,}',xlabel=label,ylabel='UAV Musanga cover (%)',xlim=(0,maxage+1),ylim=(0,100))
  rows.append(dict(site=s,variable=col,n=len(q),n_total=len(g),n_valid_history=int(g.history_valid.sum()),rho=rho))
 fig.savefig(OUT/f'{col}.png',dpi=220);plt.show();plt.close(fig)
names={1:'Undisturbed TMF',2:'Degraded TMF',3:'Deforested land',4:'Regrowth',5:'Water',6:'Other land cover'}
fig,axs=plt.subplots(2,2,figsize=(12,8),sharey=True,layout='constrained')
for s,ax in zip(SITES,axs.ravel()):
 g=J9[J9.site.eq(s)&J9.history_valid];groups=[];labels=[]
 for k,v in names.items():
  q=g[g.jrc_state.eq(k)].cover_pp
  if len(q):groups.append(q.to_numpy());labels.append(f'{v}\n(n={len(q):,})')
 if groups:ax.boxplot(groups,tick_labels=labels,showfliers=True)
 ax.tick_params(axis='x',labelrotation=30,labelsize=8);ax.set(title=s,ylabel='UAV Musanga cover (%)',ylim=(0,100))
fig.savefig(OUT/'JRC_state_cover.png',dpi=220);plt.show();plt.close(fig)
summary=pd.DataFrame(rows);summary.to_csv(OUT/'age_scatter_summary.csv',index=False);display(summary)
for d,col in [(H,'loss_status'),(J9,'deforestation_status'),(J9,'regrowth_status'),(J9,'jrc_state')]:
 q=d.groupby(['site',col],dropna=False).cover_pp.agg(n='size',mean_cover='mean',median_cover='median').reset_index()
 q.to_csv(OUT/f'{col}_summary.csv',index=False);display(q.round(2))
atomic_json(OUT/'completion.json',dict(complete=True,config=CFG,n_Hansen=len(H),n_JRC=len(J9),outputs=sorted(p.name for p in OUT.iterdir())))
print('NB09 NATIVE GRID COMPLETE',OUT)
